# C. YOLO11 Custom Backbone Skeleton (High Difficulty)

목표: Ultralytics YOLO11에 외부 backbone(ViT / Swin / ConvNeXt / InternImage) 연결 스케치.

- 완성 학습본이 아니라 인터페이스/스모크 테스트 스켈레톤
- 빠른 성능 비교는 B안(MMDetection) 권장


In [4]:
from pathlib import Path
import torch
import torch.nn as nn

PROJECT_DIR = Path.cwd().parent
CUSTOM_DIR = PROJECT_DIR / 'custom_yolo11'
CUSTOM_DIR.mkdir(parents=True, exist_ok=True)
print(CUSTOM_DIR)


<PROJECT_ROOT>/custom_yolo11


In [5]:
# Backbone stubs that return multi-scale features (~stride 8/16/32)

class ConvNeXtTinyStub(nn.Module):
    def __init__(self, in_ch=3, out_indices=(1, 2, 3)):
        super().__init__()
        self.out_indices = out_indices
        dims = [96, 192, 384, 768]
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, dims[0], 4, stride=4),
            nn.BatchNorm2d(dims[0]),
            nn.GELU(),
        )
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dims[i], dims[i], 3, padding=1, groups=dims[i]),
                nn.GELU(),
                nn.Conv2d(dims[i], dims[min(i + 1, 3)], 1),
                nn.BatchNorm2d(dims[min(i + 1, 3)]),
                nn.MaxPool2d(2) if i < 3 else nn.Identity(),
            )
            for i in range(4)
        ])
        self.out_channels = [dims[i] for i in out_indices]

    def forward(self, x):
        x = self.stem(x)
        feats = []
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i in self.out_indices:
                feats.append(x)
        return feats


class SwinTinyStub(nn.Module):
    def __init__(self, in_ch=3, out_indices=(1, 2, 3)):
        super().__init__()
        self.out_indices = out_indices
        dims = [96, 192, 384, 768]
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, dims[0], 4, stride=4),
            nn.BatchNorm2d(dims[0]),
            nn.GELU(),
        )
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dims[i], dims[min(i + 1, 3)], 3, stride=2 if i < 3 else 1, padding=1),
                nn.BatchNorm2d(dims[min(i + 1, 3)]),
                nn.GELU(),
            )
            for i in range(4)
        ])
        self.out_channels = [dims[min(i + 1, 3)] for i in out_indices]

    def forward(self, x):
        x = self.stem(x)
        feats = []
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i in self.out_indices:
                feats.append(x)
        return feats


class ViTDetStub(nn.Module):
    def __init__(self, in_ch=3, embed_dim=384, patch=16, out_indices=(0, 1, 2)):
        super().__init__()
        self.out_indices = out_indices
        self.patch = nn.Conv2d(in_ch, embed_dim, kernel_size=patch, stride=patch)
        self.blocks = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, 3, padding=1),
            nn.GELU(),
        )
        self.down1 = nn.Conv2d(embed_dim, embed_dim, 3, stride=2, padding=1)
        self.down2 = nn.Conv2d(embed_dim, embed_dim, 3, stride=2, padding=1)
        self.out_channels = [embed_dim, embed_dim, embed_dim]

    def forward(self, x):
        x = self.patch(x)
        x = self.blocks(x)
        f0, f1, f2 = x, self.down1(x), self.down2(self.down1(x))
        feats = [f0, f1, f2]
        return [feats[i] for i in self.out_indices]


BACKBONE_ZOO = {
    'convnext_t_stub': ConvNeXtTinyStub,
    'swin_t_stub': SwinTinyStub,
    'vit_stub': ViTDetStub,
}

x = torch.randn(1, 3, 640, 640)
for name, cls in BACKBONE_ZOO.items():
    m = cls()
    feats = m(x)
    print(name, '->', [tuple(f.shape) for f in feats])


convnext_t_stub -> [(1, 384, 40, 40), (1, 768, 20, 20), (1, 768, 20, 20)]
swin_t_stub -> [(1, 384, 40, 40), (1, 768, 20, 20), (1, 768, 20, 20)]
vit_stub -> [(1, 384, 40, 40), (1, 384, 20, 20), (1, 384, 10, 10)]


In [6]:
# Save backbone stub module
(CUSTOM_DIR / 'backbones.py').write_text(
    "import torch.nn as nn\n\n"
    "# Copy the stub classes from previous cell here for project use.\n"
    "print('Fill with ConvNeXtTinyStub / SwinTinyStub / ViTDetStub')\n",
    encoding='utf-8'
)
print('wrote', CUSTOM_DIR / 'backbones.py')


wrote <PROJECT_ROOT>/custom_yolo11/backbones.py


In [7]:
# Inspect YOLO11 module tree (integration entry point)
try:
    from ultralytics import YOLO
    model = YOLO('yolo11n.pt')
    print('YOLO loaded:', type(model.model))
    for i, m in enumerate(list(model.model.model)[:8]):
        print(i, type(m).__name__)
except Exception as e:
    print('Ultralytics inspect skipped:', e)

print('Integration checklist:')
print('1) backbone outs channels == neck in channels')
print('2) strides ~ 8/16/32')
print('3) num_classes = 1 (stenosis)')
print('4) freeze backbone first')


YOLO loaded: <class 'ultralytics.nn.tasks.DetectionModel'>
0 Conv
1 Conv
2 C3k2
3 Conv
4 C3k2
5 Conv
6 C3k2
7 Conv
Integration checklist:
1) backbone outs channels == neck in channels
2) strides ~ 8/16/32
3) num_classes = 1 (stenosis)
4) freeze backbone first


In [8]:
# Toy detector: backbone + tiny FPN + head (smoke only)

class TinyFPN(nn.Module):
    def __init__(self, in_channels, out_ch=128):
        super().__init__()
        self.laterals = nn.ModuleList([nn.Conv2d(c, out_ch, 1) for c in in_channels])
        self.outs = nn.ModuleList([nn.Conv2d(out_ch, out_ch, 3, padding=1) for _ in in_channels])

    def forward(self, feats):
        laterals = [l(f) for l, f in zip(self.laterals, feats)]
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i - 1] = laterals[i - 1] + nn.functional.interpolate(
                laterals[i], size=laterals[i - 1].shape[-2:], mode='nearest'
            )
        return [o(f) for o, f in zip(self.outs, laterals)]


class StubDetector(nn.Module):
    def __init__(self, backbone_name='convnext_t_stub', num_classes=1):
        super().__init__()
        self.backbone = BACKBONE_ZOO[backbone_name]()
        self.neck = TinyFPN(self.backbone.out_channels)
        self.head = nn.Conv2d(128, num_classes + 5, 1)

    def forward(self, x):
        feats = self.backbone(x)
        feats = self.neck(feats)
        return [self.head(f) for f in feats]


det = StubDetector('swin_t_stub')
y = det(torch.randn(2, 3, 640, 640))
print('toy outputs:', [tuple(t.shape) for t in y])


toy outputs: [(2, 6, 40, 40), (2, 6, 20, 20), (2, 6, 20, 20)]


## Train cells

C안은 두 트랙입니다.
1. **실용 트랙**: 기존 YOLO11n으로 pipeline 학습 (커스텀 backbone 전 baseline)
2. **스켈레톤 트랙**: StubDetector toy training (인터페이스 검증용)


In [9]:
# 1) Practical YOLO11n train (baseline while custom backbone is WIP)
from ultralytics import YOLO
import yaml

DATA_DIR = PROJECT_DIR / 'data'
YOLO_DIR = DATA_DIR / 'yolo'
RUNS_DIR = PROJECT_DIR / 'runs' / 'detect'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Choose one pipeline for C-track baseline
PIPE = 'clahe_gamma'   # original | clahe | clahe_gamma
MODEL_CKPT = 'yolo11n.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0

base = YOLO_DIR / PIPE
assert (base / 'images' / 'train').exists(), base
assert (base / 'images' / 'val').exists(), base

data_yaml = base / 'data.yaml'
data = {
    'path': str(base),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['stenosis'],
}
with open(data_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print('data.yaml:', data_yaml)

model = YOLO(MODEL_CKPT)
res = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f'yolo11n_Ctrack_{PIPE}',
    exist_ok=True,
    pretrained=True,
    val=True,
    plots=True,
)
print('saved to:', res.save_dir)


Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
50 epochs completed in 0.065 hours.
Validating <PROJECT_ROOT>/runs/detect/yolo11n_Ctrack_clahe_gamma/weights/best.pt...
                   all        200        406      0.346      0.404      0.296      0.122
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
Results saved to <PROJECT_ROOT>/runs/detect/yolo11n_Ctrack_clahe_gamma
saved to: <PROJECT_ROOT>/runs/detect/yolo11n_Ctrack_clahe_gamma


In [10]:
# 2) Toy train loop for StubDetector (interface smoke test ONLY)
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
backbone_name = 'swin_t_stub'  # convnext_t_stub | swin_t_stub | vit_stub

# fake images / fake targets (not real detection loss)
N, C, H, W = 32, 3, 320, 320
x = torch.randn(N, C, H, W)
y = torch.randn(N, 1)  # dummy

loader = DataLoader(TensorDataset(x, y), batch_size=4, shuffle=True)

class StubDetTrainable(nn.Module):
    def __init__(self, backbone_name):
        super().__init__()
        self.backbone = BACKBONE_ZOO[backbone_name]()
        self.neck = TinyFPN(self.backbone.out_channels, out_ch=128)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 1)

    def forward(self, imgs):
        feats = self.backbone(imgs)
        feats = self.neck(feats)
        # use highest-res feature for toy regression
        z = self.pool(feats[0]).flatten(1)
        return self.fc(z)

model = StubDetTrainable(backbone_name).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model.train()
for epoch in range(3):
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += float(loss.item())
    print(f'epoch {epoch+1}: loss={total/len(loader):.4f}')

ckpt = CUSTOM_DIR / f'stub_{backbone_name}.pt'
torch.save({'model': model.state_dict(), 'backbone': backbone_name}, ckpt)
print('toy checkpoint saved:', ckpt)
print('NOTE: this does NOT train real stenosis detection. Use YOLO cell above for real training.')


epoch 1: loss=4.5122
epoch 2: loss=2.0322
epoch 3: loss=1.2110
toy checkpoint saved: <PROJECT_ROOT>/custom_yolo11/stub_swin_t_stub.pt
NOTE: this does NOT train real stenosis detection. Use YOLO cell above for real training.


In [11]:
# 3) Custom backbone integration status checklist
print('Real YOLO11 backbone replacement needs:')
print('- register custom module in ultralytics parse_model')
print('- match neck in-channels to backbone out-channels')
print('- keep strides near 8/16/32')
print('- then train with model.train(data=...)')
print('\nUntil then:')
print('- use B notebook (MMDet Swin/ConvNeXt) for backbone comparison')
print('- use YOLO train cell above for pipeline baseline')


Real YOLO11 backbone replacement needs:
- register custom module in ultralytics parse_model
- match neck in-channels to backbone out-channels
- keep strides near 8/16/32
- then train with model.train(data=...)

Until then:
- use B notebook (MMDet Swin/ConvNeXt) for backbone comparison
- use YOLO train cell above for pipeline baseline


## C안 다음 액션
1. B안에서 Swin/ConvNeXt가 YOLO11을 넘는지 확인
2. 이득 있으면 Ultralytics fork로 parse_model 등록
3. InternImage는 DCNv3 이슈로 마지막

우선순위: **ConvNeXt-T / Swin-T → ViT → InternImage**
